# FBI Crime Data Explorer — the API

What the endpoints are, what they return, and the three ways they mislead. The
client in [`client.ipynb`](client.ipynb) wraps all of this; read here first if
something in the data looks wrong.

**The live base is `https://api.usa.gov/crime/fbi/cde`.** The older
`crime/fbi/sapi` service that most documentation and every tutorial still
describes returns 404, and the published Swagger documents that dead service.
Everything below was established by probing.

Authentication is the shared **api.data.gov** key (`DATA_GOV_KEY` in `.env`),
which opens every service behind that gateway rather than the FBI alone. It
goes in the query string as `API_KEY` — capitalised, since these routes ignore
the lowercase spelling other api.data.gov services accept. `DEMO_KEY` allows
about thirty calls an hour, a real key a thousand.

In [1]:
import sys
from pathlib import Path

# Anchor on this notebook's own directory rather than on the working directory:
# VS Code can root a notebook at the workspace folder instead of at the file,
# and then a relative "../" resolves somewhere else entirely and the imports
# below fail with ModuleNotFoundError.
_here = Path.cwd()
_FBI = _here if (_here / "utils" / "mcp.py").exists() else _here / "notebooks" / "fbi"
for _path in (str(_FBI), str(_FBI.parents[1])):      # notebooks/fbi, then the meida root
    if _path not in sys.path:
        sys.path.insert(0, _path)

import asyncio
from collections import Counter

from clients import FbiClient
from clients.fbi import OFFENSES
from environment import get_fbi_base_url

print("base:", get_fbi_base_url())
print("offence slugs:", len(OFFENSES))
print(" ", ", ".join(OFFENSES))

base: https://api.usa.gov/crime/fbi/cde
offence slugs: 10
  violent-crime, homicide, rape, robbery, aggravated-assault, property-crime, burglary, larceny, motor-vehicle-theft, arson


## What one call returns

`summarized/{national|state/ST}/{offense}` is the whole crime surface. One
request carries more than the observations: both measures, both units, the
denominator and the reporting completeness.

In [2]:
async with FbiClient() as fbi:
    payload = await fbi.get_offenses("homicide", state="DC", start="01-2015", end="12-2024")

print(f"{payload.offense} / {payload.scope} — vintage {payload.max_data_date}, "
      f"refreshed {payload.last_refresh_date}\n")
for series in payload.series:
    span = series.span
    print(f"  {series.label:32} {series.measure:11} {str(series.unit):17} "
          f"{len(series.observations):4} points  {span[0]}..{span[1]}")

homicide / DC — vintage 09/2026, refreshed 09/15/2026

  District of Columbia Offenses    offenses    count              120 points  01-2015..12-2024
  District of Columbia Clearances  clearances  count              120 points  01-2015..12-2024
  District of Columbia Offenses    offenses    rate per 100,000   120 points  01-2015..12-2024
  District of Columbia Clearances  clearances  rate per 100,000   120 points  01-2015..12-2024
  District of Columbia             value       people             120 points  01-2015..12-2024
  District of Columbia             value       percent            120 points  01-2015..12-2024


Ten series from one request: the scope's offences and clearances as counts and
as rates, the national comparison for both, the population denominator, and the
coverage percentage.

**Clearances are the useful surprise.** Clearances ÷ offences is the clearance
rate — the standard measure of enforcement intensity — and it arrives with the
crime data rather than needing a separate source.

## Coverage: the NIBRS break, as a number

Agencies moved from Summary UCR to NIBRS through 2021 and many stopped
reporting during the transition. The API publishes what share of the population
was covered each month, so the break is measurable instead of a footnote.

In [3]:
async with FbiClient() as fbi:
    national = await fbi.get_offenses("violent-crime", start="01-2015", end="12-2024")

coverage = next(s for s in national.series if s.unit == "percent")
by_year = {}
for observation in coverage.observations:
    if observation.value is not None:
        by_year.setdefault(observation.period.split("-")[1], []).append(observation.value)

print("percent of the US population covered by reporting agencies")
for year, values in sorted(by_year.items()):
    mean = sum(values) / len(values)
    print(f"  {year}  {mean:5.1f}  " + "#" * int(mean / 2))

percent of the US population covered by reporting agencies
  2015   97.6  ################################################
  2016   97.7  ################################################
  2017   96.5  ################################################
  2018   97.5  ################################################
  2019   96.2  ################################################
  2020   95.9  ###############################################
  2021   77.0  ######################################
  2022   94.8  ###############################################
  2023   95.9  ###############################################
  2024   96.4  ################################################


96–98% in normal years, **74% in 2021**. A 2021 value is not comparable with a
2019 one, and any model using this data should weight or flag by this series
rather than treating the dip as a fall in crime.

**Why 2021.** The FBI retired the Summary Reporting System (SRS) on 1 January
2021 in favour of the **National Incident-Based Reporting System (NIBRS)**, and
agencies that had not converted dropped out of the counts.

**And the counting rule changed with it.** SRS applied the *hierarchy rule* —
one incident contributed only its most serious offence. NIBRS records every
offence in an incident, along with victims, offenders, weapons and location. A
robbery involving an assault is one offence before 2021 and two after. Coverage
recovering to 96% does not undo that, and nothing in the API flags it, so a
series spanning 2021 is spliced rather than continuous.


In [4]:
# 1. An agency request returns THREE geographies, widest first.
async with FbiClient() as fbi:
    raw = await fbi.get_offenses("homicide", ori="NY0303000",
                                 include_comparisons=True, start="01-2019", end="12-2019")
    filtered = await fbi.get_offenses("homicide", ori="NY0303000",
                                      start="01-2019", end="12-2019")

print("everything the payload carries:")
for series in raw.series:
    if series.measure in ("offenses", "clearances"):
        print(f"   {series.label:46} {series.unit}")
print("\nwhat the client returns by default:")
for series in filtered.series:
    if series.measure in ("offenses", "clearances"):
        print(f"   {series.label:46} {series.unit}")

everything the payload carries:
   New York City Police Department Offenses       count
   New York City Police Department Clearances     count
   New York Offenses                              rate per 100,000
   New York Clearances                            rate per 100,000
   United States Offenses                         rate per 100,000
   United States Clearances                       rate per 100,000
   New York City Police Department Offenses       rate per 100,000
   New York City Police Department Clearances     rate per 100,000

what the client returns by default:
   New York City Police Department Offenses       count
   New York City Police Department Clearances     count
   New York City Police Department Offenses       rate per 100,000
   New York City Police Department Clearances     rate per 100,000


In [5]:
# 2. pe/national and pe/state answer 200 with the full shape and nothing in it.
async with FbiClient() as fbi:
    import httpx
    key = fbi._api_key
    async with httpx.AsyncClient(base_url=get_fbi_base_url(), timeout=30) as raw:
        national = (await raw.get("pe/national",
                                  params={"from": "1985", "to": "2024", "API_KEY": key})).json()
filled = sum(1 for block in national.values() if isinstance(block, dict)
             for series in block.values() if isinstance(series, dict)
             for value in series.values() if value is not None)
print("pe/national blocks:", list(national))
print("non-null values in the whole payload:", filled, "— employment is agency-level only")

pe/national blocks: ['rates', 'actuals', 'tooltips', 'populations', 'cde_properties']
non-null values in the whole payload: 2 — employment is agency-level only


In [6]:
# 3. The rates block of pe divides by a participated population that can be zero.
async with FbiClient() as fbi:
    employment = await fbi.get_employment("DCMPD0000")

officers = employment.officers
print("Washington PD, sworn officers from the counts:")
for year in (1985, 1990, 2000, 2010, 2019, 2024):
    print(f"  {year}  {officers[year]:,}")
print("\nUse the counts and bring your own denominator; the API's own rate is "
      "unusable where its population base is wrong or zero.")

Washington PD, sworn officers from the counts:
  1985  3,837
  1990  4,740
  2000  3,628
  2010  3,945
  2019  3,809
  2024  3,276

Use the counts and bring your own denominator; the API's own rate is unusable where its population base is wrong or zero.


## The agency registry

`agency/byStateAbbr/{ST}` is the only route that enumerates agencies, and it
carries the per-agency NIBRS start date — the transition dated locally rather
than inferred from the national dip.

In [7]:
async with FbiClient() as fbi:
    dc = await fbi.get_agencies("DC")

for agency in dc:
    if agency.agency_name:
        print(f"  {agency.ori}  {agency.agency_name[:46]:46} {agency.agency_type_name:8} "
            f"nibrs={agency.is_nibrs} {agency.nibrs_start_date or ''}")

  DC0011300  District of Columbia Fire and Emergency Medica Other    nibrs=False 
  DCMPD0000  Washington Police Department                   City     nibrs=True 2021-08-01
  DCMTP0000  Metro Transit Police                           Other    nibrs=True 2000-01-01


Nationally that is 51 calls for 19,636 agencies, of which 11,784 are city
departments. Enumerating them is cheap; pulling employment for all of them is
not — 19,636 calls against a 1,000-per-hour allowance is about twenty hours, so
take the cities you intend to model.

Next: [`client.ipynb`](client.ipynb) for the series themselves.

## What a catalog entry looks like

The CDE has no metadata endpoint — `lookup/offenses` answers
`{"crimeGroups": null}` for every type tried — but it does not need one. Each
call carries the span, the denominator, the coverage and the vintage alongside
the observations, so the catalog is a by-product of reading the data rather
than a second pull.

`utils.catalog` turns one payload into entries. Here is a single series, whole.

In [8]:
import sys
from pathlib import Path

# Anchor on this notebook's own directory rather than on the working directory:
# VS Code can root a notebook at the workspace folder instead of at the file,
# and then a relative "../" resolves somewhere else entirely and the imports
# below fail with ModuleNotFoundError.
_here = Path.cwd()
_FBI = _here if (_here / "utils" / "mcp.py").exists() else _here / "notebooks" / "fbi"
for _path in (str(_FBI), str(_FBI.parents[1])):      # notebooks/fbi, then the meida root
    if _path not in sys.path:
        sys.path.insert(0, _path)

import yaml
from utils import catalog

entries = await catalog.harvest(offenses=("homicide",), scopes=("DCMPD0000",),
                                start="01-1985", end="12-2024")
offences_entry = next(e for e in entries if e["facets"]["measure"] == "offenses")
print(yaml.safe_dump(offences_entry, sort_keys=False, allow_unicode=True, width=86))

series_id: fbi/homicide/DCMPD0000/offenses
concept: homicide
title: Homicide offenses, Washington Police Department, monthly
unit: count
frequency: Monthly
provisional: false
is_active: true
facets:
  geography: Washington Police Department
  scope: DCMPD0000
  offense: homicide
  measure: offenses
  agency_type: City
  state: DC
  county: NOT SPECIFIED
observation_start: '1985-01-01'
observation_end: '2024-12-01'
definition: Murder and non-negligent manslaughter. Excludes negligent manslaughter, justifiable
  homicide and attempts. Offences known to law enforcement.
coverage_mean_percent: 100.0
coverage_min_percent: 100.0
sources:
- kind: fbi_cde
  endpoint: summarized/DCMPD0000/homicide
  vintage: 09/2026
  refreshed: 09/15/2026
retrieval:
  tool: fbi_offenses
  offense: homicide
  scope: DCMPD0000
nibrs_start_date: '2021-08-01'



Where each field comes from:

| Source | Fields |
| --- | --- |
| The payload | `observation_start`/`_end`, `coverage_*`, `sources.vintage`, and the geography in `facets` — an ORI resolves to the department's own name |
| The request | `series_id`, `concept`, `title`, `unit`, `frequency`, `retrieval` |
| The agency registry | `agency_type`, `state`, `county`, and **`nibrs_start_date`** |
| Written by hand | `definition` — the API has no dictionary, and what a clearance means (arrest *or exceptional means*) is the caveat people miss |
| Generated later | `description`, for semantic search, by `db_import.descriptions` |

Two of those make the entry self-describing about its own reliability.
`coverage_min_percent` is 100 here and 74.12 nationally, because Washington PD
reported throughout while a fifth of the country did not in 2021 — so this
series crosses that year cleanly and the national one does not. And
`nibrs_start_date` is 2021-08-01, which is where *this* agency's reporting
break falls, rather than wherever the national dip suggests.

`retrieval` points at the MCP tool rather than at stored series: the FBI is a
live source like FRED and BIS, so nothing here lands in Postgres.